# Exercise: K-means vs Gaussian Mixtures

In [ ]:
import os
from urllib.request import urlretrieve
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from scipy.stats import multivariate_normal, wishart
from scipy.spatial import Voronoi, voronoi_plot_2d
from cycler import cycler
import seaborn as sns

# Set the color scheme
sns.set_theme()
colors = [
    "#0076C2",
    "#EC6842",
    "#A50034",
    "#009B77",
    "#FFB81C",
    "#E03C31",
    "#6CC24A",
    "#EF60A3",
    "#0C2340",
    "#00B8C8",
    "#6F1D77",
]
plt.rcParams["axes.prop_cycle"] = cycler(color=colors)

In this exercise, we will compare two different clustering algorithms: $K$-means clustering and Gaussian mixture models.
We will use the 'old faithful' dataset, to compare these two methods
This dataset is based on measurements of the [old faithful geyser](https://en.wikipedia.org/wiki/Old_Faithful) in Yellowstone National Park, known for having very regular eruptions.
In the figure below, the eruption durations are plotted against the waiting times until the next eruption.
As usual, we have standardized the data as well to improve the performance of our clustering algorithms.

In [ ]:
# Download the old faithful dataset (if necessary)
url = "https://surfdrive.surf.nl/s/PyDTFPQeJ5NsHPX/download"
filename = "old_faithful.txt"

if not os.path.isfile(filename):
    print(f"Downloading {filename}...")
    urlretrieve(url, filename)

# load the old faithful dataset
data_all = np.genfromtxt(filename, dtype=float, skip_header=26, usecols=(1, 2))

# Remove the final point from the data, which is an outlier
data_all = data_all[:-1, :]

# Standardize the data
scaler = StandardScaler()
data_sc = scaler.fit_transform(data_all)

In [ ]:
# Plot the ground truth, data and predictions
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(8, 4), tight_layout=True)
ax1.scatter(data_all[:, 0], data_all[:, 1], color="C0", marker="x")
ax1.set_xlim((1.2, 5.8))
ax1.set_ylim((37, 102))
ax1.set_title("Original data")
ax1.set_xlabel("duration of eruption (s)")
ax1.set_ylabel("waiting time until next eruption (s)")

ax2.scatter(data_sc[:, 0], data_sc[:, 1], color="C0", marker="x")
ax2.set_xlim((-2.3, 2.3))
ax2.set_ylim((-2.3, 2.3))
ax2.set_title("Standardized data")
ax2.set_xlabel("duration of eruption")
ax2.set_ylabel("waiting time until next eruption")
plt.show()

## K-means clustering

We will first turn to $K$-means clustering.
We aim to categorize our $N$ data points systematically into $K$ clusters.
Each data point $\boldsymbol x_n$ is assigned to one of the $K$ clusters, and each cluster has its center $\boldsymbol \mu_k$.
Of course, we want to choose the assignment of the data points to the clusters and the location of their centers $\boldsymbol \mu_k$ in some optimal way to get the best results.
To do this, we define the following loss function $J$:

$$
J = \sum_{n=1}^N \sum_{k=1}^K r_{nk} \| \boldsymbol x_n - \boldsymbol \mu_k \|^2
$$

where

$$
r_{nk} = 
\begin{cases}
    1 \text{ if $\boldsymbol x_n$ is in cluster $k$} \\
    0 \text{ if $\boldsymbol x_n$ is not in cluster $k$}
\end{cases}
$$

There are two components to this loss function that can be updated separately.
First, we can fix $\boldsymbol \mu_k$ and choose the cluster assignment $r_{nk}$ such that it minimizes $J$.
Then, we can fix the $r_{nk}$ and move the cluster centers $\boldsymbol \mu_k$ to the optimal locations.
This process is repeated until no data points are reassigned to new clusters, and the clusters no longer move around.

Now, let's think about how to do each of these optimizations.
The optimization of the first step (determining the cluster assignments $r_{nk}$) is quite straightforward:
we get the biggest loss reduction if we assign each datapoint $\boldsymbol x_n$ to whichever cluster has the closest center $\boldsymbol \mu_k$.
The optimization of the second step (determining the cluster centers $\boldsymbol \mu_k$) can be derived by setting the gradient $\frac{\partial J}{\partial \boldsymbol \mu_k} = 0$.
Doing this yields the following optimal expression for each cluster center $\boldsymbol \mu_k$:
$$
\boldsymbol \mu_k = \frac{\sum_{n=1}^N r_{nk} \boldsymbol x_n}{\sum_{n=1}^N r_{nk}}
$$

See if you can derive this expression yourself!

In the code block below, a start has been made to an implementation of the K-means algorithm.
Your task is to complete the code by implementing the two steps laid out above.

In [ ]:
def K_means(X, K, mu_init, max_iter=100):
    N = X.shape[0]

    # clusters is a vector storing for each point x_n which cluster k it belongs to
    clusters = np.zeros(N, dtype=int)

    # Get the initial cluster centers
    mu = mu_init

    # Store the centers, clustering and loss from each iteration
    mu_list = []
    clusters_list = []
    loss_list = []

    for i in range(max_iter):
        # Check in each iteration if points have been assigned to new clusters
        point_reassigned = False

        ###########################################
        # STEP 1: optimize the cluster assignment #
        ###########################################
        for n in range(N):
            x_n = X[n, :]

            # ---------------------- student exercise --------------------------------- #
            # YOUR CODE HERE
            # ---------------------- student exercise --------------------------------- #

            # Check if the cluster has been updated
            if clusters[n] != new_cluster:
                point_reassigned = True

            # Store the new cluster
            clusters[n] = new_cluster

        # If no point have been assigned to new clusters, end the algorithm
        if not point_reassigned:
            break

        # Recompute the cluster index matrix
        r_nk = np.zeros((N, K))
        for n in range(N):
            k = clusters[n]
            r_nk[n, k] = 1

        # Compute the loss
        loss = 0.0
        for n in range(N):
            for k in range(K):
                loss += r_nk[n, k] * sum((X[n, :] - mu[k,:]) ** 2)

        # Store the centers, clusters and loss
        mu_list.append(mu.copy())
        clusters_list.append(clusters.copy())
        loss_list.append(loss)

        ########################################
        # STEP 2: optimize the cluster centers #
        ########################################
        for k in range(K):
            # ---------------------- student exercise --------------------------------- #
            # YOUR CODE HERE
            # ---------------------- student exercise --------------------------------- #

            mu[k, :] = new_center

    return mu_list, clusters_list, loss_list

We are now going to run the $K$-means algorithm you just coded on the old faithful dataset.
We will set $K=2$ and initialize $\boldsymbol \mu_k$ randomly using a multivariate normal distribution.

In [ ]:
# Get the number of clusters K and dimensionality D
K = 2
D = data_sc.shape[1]

# Initialize the cluster centers in random locations
rng = np.random.default_rng(seed=0)
mu_init = rng.multivariate_normal(np.zeros(D), np.identity(D), size=(K))

# Run the K-means algorithm
mu_list, clusters_list, loss_list = K_means(data_sc, K, mu_init)

cmap = mpl.colors.ListedColormap(colors[:K])
norm = mpl.colors.Normalize(0, K)

# Plot the results
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
    nrows=2, ncols=2, figsize=(8, 8), sharex=True, sharey=True, tight_layout=True
)
for i, ax in zip([0, 1, 2, -1], [ax1, ax2, ax3, ax4]):
    mu_i = mu_list[i]
    clusters_i = clusters_list[i]

    # Remove the nan values from the cluster center list
    nan_idx = np.isnan(mu_i[:, 0])
    mu_i = mu_i[~nan_idx, :]
    c = np.arange(K)[~nan_idx]

    # Plot the data and the cluster means
    ax.scatter(
        mu_i[:, 0],
        mu_i[:, 1],
        marker="o",
        c=c,
        cmap=cmap,
        norm=norm,
        label="cluster centers",
    )
    ax.scatter(
        data_sc[:, 0],
        data_sc[:, 1],
        marker="x",
        c=clusters_i,
        cmap=cmap,
        norm=norm,
        alpha=0.4,
        label="clustered data",
    )

    # Set the axis title
    if ax is ax4:
        ax.set_title("iteration {:d} (final)".format(len(loss_list) - 1))
    else:
        ax.set_title("iteration {:d}".format(i))

    # Set the axis labels
    if ax is ax3 or ax is ax4:
        ax.set_xlabel("duration")
    if ax is ax1 or ax is ax3:
        ax.set_ylabel("waiting time")

    # Plot the decision boundary
    vor = Voronoi(
        np.concatenate(
            (mu_i, np.array([[-1000, 0], [0, 1000], [1000, 0], [0, -1000]])), axis=0
        )
    )
    voronoi_plot_2d(vor, ax, show_points=False, show_vertices=False, line_alpha=0.5)

    # Set the axis limits
    ax.set_xlim((-2.3, 2.3))
    ax.set_ylim((-2.3, 2.3))

ax4.legend(loc="lower right")
plt.show()

In the figure above, the cluster centers move toward the main clusters in the data as the algorithm progresses.
Play around with the number of clusters `K` and initial center location `mu_init` and ask yourself the following questions:
- If you choose a large number of clusters, like `K = 15`, what do you notice about the size and shape of the clusters?
- How do the initial center locations affect the final results?
- How do the initial center locations affect the number of iterations until convergence?

## Gaussian mixtures

You may notice that $K$-means assigns a hard boundary between the clusters.
This feature might actually be undesirable, since it provides no information about points that are close to the boundary and cannot clearly be categorized as either cluster A or cluster B.
One approach that does allow for this nuance is a Gaussian mixture model.
The idea is similar to $K$-means, but instead of assigning each data point to a single category, each datapoint is assigned a probability of belonging to each category.
The Gaussian mixture distribution is given by:

$$
p(\boldsymbol x_n) = \sum_{k=1}^K \pi_k \mathcal{N}(\boldsymbol x_n | \boldsymbol \mu_k, \boldsymbol \Sigma_k)
$$

In other words, it is a sum of $k$ Gaussian distributions, each of which has its own mean $\boldsymbol \mu_k$ and covariance matrix $\boldsymbol \Sigma_k$, weighted by $\pi_k$.

We now introduce the latent variable $\boldsymbol z$, a $K$-dimensional vector, of which one element is $1$ and all other elements are $0$.
The probability of the $k$-th element being equal to $1$ is given by the mixing coefficients $\pi_k$:

$$
p(z_k = 1) = \pi_k 
$$

If we want to draw a sample from our Gaussian mixture model, we first sample $\boldsymbol z$, which determines the cluster to which our sample belongs.
We then sample from the Gaussian with mean $\boldsymbol \mu_k$ and covariance matrix $\boldsymbol \Sigma_k$ associated with this cluster.
This means that the conditional distribution of $\boldsymbol x_n$ given $\boldsymbol z$ is given by:

$$
p(\boldsymbol x_n | z_k = 1) = \mathcal{N}(\boldsymbol x_n | \boldsymbol \mu_k, \boldsymbol \Sigma_k)
$$

Try to draw the graph model of the Gaussian mixture model.
Also, see if you can prove that by marginalizing over $\boldsymbol z$, you indeed reobtain the Gaussian mixtures formula stated above:

$$
p(\boldsymbol x_n) = \sum_{\boldsymbol z} p(\boldsymbol z) p(\boldsymbol x_n | \boldsymbol z) = \sum_{k=1}^K \pi_k \mathcal{N}(\boldsymbol x_n | \boldsymbol \mu_k, \boldsymbol \Sigma_k)
$$

Finally, using Bayes' theorem, we can compute the probability of $\boldsymbol z$ given $\boldsymbol x_n$.
Since this quantity will pop up a few times later on, we give it a short notation $\gamma(z_k)$:

$$
\gamma(z_{nk}) = p(z_k = 1 | \boldsymbol x_n) = \frac{p(z_k = 1) p(\boldsymbol x_n | z_k = 1)}{p(\boldsymbol x_n)} = \frac{\pi_k \mathcal{N}(\boldsymbol x_n | \boldsymbol \mu_k, \boldsymbol \Sigma_k)}{\sum_{j=1}^K \pi_j \mathcal{N}(\boldsymbol x_n | \boldsymbol \mu_j, \boldsymbol \Sigma_j)}
$$

This quantity $\gamma(z_{nk})$ is sometimes referred to as the *responsibility* that cluster $k$ has for explaining observation $\boldsymbol x_n$.
Another way to look at $\gamma(z_{nk})$ is as a probabilistic version of the $r_{nk}$ variable that we saw in the $K$-means model.

### EM for Gaussian mixtures

We will employ a maximum likelihood approach to fit a Gaussian mixture model to our data.
The log-likelihood of the data given our model parameters is given by:

$$
\ln p(\boldsymbol X | \boldsymbol \pi, \boldsymbol \mu, \boldsymbol \Sigma) = \sum_{n=1}^N \ln \left\{\sum_{k=1}^K \pi_k \mathcal{N}(\boldsymbol x_n | \boldsymbol \mu_k, \boldsymbol \Sigma_k) \right\}
$$

To maximize the log-likelihood, we will use the expectation maximization (EM) algorithm.
This algorithm consists of two steps:
1. The expectation step (E step): we fix the parameters $\boldsymbol \mu_k$, $\boldsymbol \Sigma_k$ and $\pi_k$, and compute the responsibilities $\gamma(z_{nk})$ for each cluster $k$ and datapoint $\boldsymbol x_n$.
2. The maximization step (M step): we fix the responsibilities $\gamma(z_{nk})$, and optimize the parameters $\boldsymbol \mu_k$, $\boldsymbol \Sigma_k$ and $\pi_k$ to maximize the log-likelihood.

The optimal values in the M step can be found by setting the derivative of the log-likelihood with respect to $\boldsymbol \mu_k$, $\boldsymbol \Sigma_k$ and $\pi_k$ to $0$.
This yields the following expressions for the new values of these parameters:

$$
\boldsymbol \mu_k = \frac{\sum_{n=1}^N \gamma(z_{nk}) \boldsymbol x_n}{\sum_{n=1}^N \gamma(z_{nk})} \\
$$

$$
\boldsymbol \Sigma_k = \frac{\sum_{n=1}^N \gamma(z_{nk}) (\boldsymbol x_n - \boldsymbol \mu_k)(\boldsymbol x_n - \boldsymbol \mu_k)^T}{\sum_{n=1}^N \gamma(z_{nk})} \\
$$

$$
\pi_k = \frac{\sum_{n=1}^N \gamma(z_{nk})}{\sum_{n=1}^N \sum_{k=1}^K \gamma(z_{nk})} \\
$$

Note again the similarity in the expression found here for $\boldsymbol \mu_k$ and the expression for the cluster centers we found before in the $K$-means algorithm.
The expression we find for $\pi_k$ also allows for a nice interpretation:
It can be seen as the average responsibility the $k$-th cluster has for explaining the data.

We will now implement our Gaussian mixture model that is fitted to the data using the EM algorithm.

In [ ]:
def EM_mixtures(X, K, mu_init, Sigma_init, pi_init, max_iter=1000, tol=1e-6):
    N = X.shape[0]
    D = X.shape[1]

    # clusters is a vector storing for each point x_n which cluster k it belongs to
    clusters = np.zeros(N, dtype=int)

    # Get the initial cluster means, covariances and weights
    mu = mu_init
    Sigma = Sigma_init
    pi = pi_init

    # Store the centers, clustering and loss from each iteration
    mu_list = []
    Sigma_list = []
    pi_list = []
    z_nk_list = []
    loglikelihood_list = []

    for i in range(max_iter):
        ########################################
        # E STEP: compute the responsibilities #
        ########################################

        z_nk = np.zeros((N, K))

        for n in range(N):
            x_n = X[n, :]

            # ---------------------- student exercise --------------------------------- #
            # YOUR CODE HERE
            # ---------------------- student exercise --------------------------------- #

        ##############################
        # Compute the log likelihood #
        ##############################

        loglikelihood = 0.0

        for n in range(N):
            x_n = X[n, :]
            likelihood_x = 0.0

            # For each datapoint, marginalize over z to get its likelihood
            for k in range(K):
                # Compute p(x|z)
                gaussian = multivariate_normal(mu[k, :], Sigma[k, :, :])
                likelihood_x_given_z = gaussian.pdf(x_n)

                # Compute p(x,z) = p(z) * p(x|z)
                joint_x_z = pi[k] * likelihood_x_given_z

                # Compute p(x) = int p(x,z) dz
                likelihood_x += joint_x_z

            # Take the log and add it to log likelihood of the full dataset
            loglikelihood += np.log(likelihood_x)

        # Check for convergence
        if len(loglikelihood_list) > 0:
            if np.abs(loglikelihood - loglikelihood_list[-1]) < tol:
                break

        # Store the means, covariances, weights, responsibilities and loss
        mu_list.append(mu.copy())
        Sigma_list.append(Sigma.copy())
        pi_list.append(pi)
        z_nk_list.append(z_nk.copy())
        loglikelihood_list.append(loglikelihood)

        ###################################
        # M STEP: optimize the parameters #
        ###################################
        for k in range(K):
            # ---------------------- student exercise --------------------------------- #
            # YOUR CODE HERE
            # ---------------------- student exercise --------------------------------- #

            # Update mu, sigma and pi
            mu[k, :] = new_mu
            Sigma[k, :, :] = new_Sigma
            pi[k] = new_pi

    return mu_list, Sigma_list, pi_list, z_nk_list, loglikelihood_list

We are now going to run the EM algorithm you just coded on the old faithful dataset.
We will again set $K=2$ and initialize $\boldsymbol \mu_k$ randomly using a multivariate normal distribution.
We initialize $\boldsymbol \Sigma_k = \boldsymbol I$ and set $\pi_k = \frac{1}{K}$ for all clusters.

In [ ]:
# Get the number of clusters K and dimensionality D
K = 2
N = data_sc.shape[0]
D = data_sc.shape[1]

# Initialize the cluster centers in random locations
rng = np.random.default_rng(seed=0)
mu_init = np.zeros((K, D))
Sigma_init = np.zeros((K, D, D))
for k in range(K):
    mu_init[k, :] = rng.multivariate_normal(np.zeros(D), np.identity(D))
    Sigma_init[k, :] = np.identity(D)
pi_init = np.ones(K) * 1 / K

# Run the EM algorithm on the Gaussian mixture model
mu_list, Sigma_list, pi_list, z_nk_list, loglikelihood_list = EM_mixtures(
    data_sc, K, mu_init, Sigma_init, pi_init
)

cmap = mpl.colors.ListedColormap(colors[:K])
norm = mpl.colors.Normalize(0, K)

# Plot the results
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
    nrows=2, ncols=2, figsize=(8, 8), sharex=True, sharey=True, tight_layout=True
)
for i, ax in zip([0, 2, 5, -1], [ax1, ax2, ax3, ax4]):
    mu_i = mu_list[i]
    Sigma_i = Sigma_list[i]
    z_nk_i = z_nk_list[i]

    # Compute the average colors
    c = np.zeros((N, 3))
    for n in range(N):
        for k in range(K):
            r, g, b = mpl.colors.to_rgb(colors[k])
            c[n, 0] += r * z_nk_i[n, k]
            c[n, 1] += g * z_nk_i[n, k]
            c[n, 2] += b * z_nk_i[n, k]

    # Plot the data and the cluster means
    ax.scatter(
        data_sc[:, 0], data_sc[:, 1], marker="x", c=c, alpha=0.4, label="clustered data"
    )
    ax.scatter(
        mu_i[:, 0],
        mu_i[:, 1],
        marker="o",
        c=np.arange(K),
        cmap=cmap,
        norm=norm,
        label="cluster centers",
    )

    # Plot the covariance contours
    for k in range(K):
        # Get the eigendecomposition of each covariance matrix
        mu_k = mu_i[k, :]
        Sigma_k = Sigma_i[k, :, :]
        l, Q = np.linalg.eigh(Sigma_k)

        # Get the contour circles from the eigendecomposition
        width = 2 * np.sqrt(l[0])
        height = 2 * np.sqrt(l[1])
        angle = np.arctan2(Q[1, 0], Q[0, 0]) / np.pi * 180

        # Add the ellipses to the plot
        ax.add_patch(
            mpl.patches.Ellipse(
                mu_k,
                width=width,
                height=height,
                angle=angle,
                fill=None,
                color=colors[k],
                alpha=0.7,
                linewidth=2,
            )
        )
        ax.add_patch(
            mpl.patches.Ellipse(
                mu_k,
                width=2 * width,
                height=2 * height,
                angle=angle,
                fill=None,
                color=colors[k],
                alpha=0.7,
                linewidth=2,
            )
        )

    # Set the axis title
    if ax is ax4:
        ax.set_title("iteration {:d} (final)".format(len(loglikelihood_list) - 1))
    else:
        ax.set_title("iteration {:d}".format(i))

    # Set the axis labels
    if ax is ax3 or ax is ax4:
        ax.set_xlabel("eruption duration")
    if ax is ax1 or ax is ax3:
        ax.set_ylabel("waiting time")

    # Set the axis limits
    ax.set_xlim((-2.3, 2.3))
    ax.set_ylim((-2.3, 2.3))

ax4.legend(loc="lower right")
plt.show()

You can see in the figure above how the Gaussian mixture model can find an appropriate clustering for the data as the EM algorithm progresses.
Play around with the number of clusters `K` and ask yourself the following questions:
- Are there any points that the Gaussian mixture model cannot adequately explain?
- Does increasing the number of clusters to `K = 3` improve the model?
- If you choose a large number of clusters, like `K = 8`, what do you notice about the size and shape of the clusters? (note: this may take a while to run)
- In your code, turn off the recomputation of $\Sigma_k$ based on the new responsibilities. How do the results you obtain now compare to the $K$-means results?